# Day 9 Data Sync and Check
Download and parse the 100-paper corpus into the `Data_V2` pipeline.

In [1]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/AIGC/Data_V2")

for p in [
    DATA_DIR / "pdfs" / "nan.pdf",
    DATA_DIR / "parsed" / "nan.json",
]:
    if p.exists():
        p.unlink()
        print("Deleted", p)
    else:
        print("Not found", p)

Deleted /content/drive/MyDrive/AIGC/Data_V2/pdfs/nan.pdf
Deleted /content/drive/MyDrive/AIGC/Data_V2/parsed/nan.json


In [2]:
from google.colab import drive
from pathlib import Path
import os, json, shutil, glob, subprocess, sys

# Ensure the mount point is clean before mounting
if os.path.exists("/content/drive") and os.listdir("/content/drive"):
    print("Mount point /content/drive not empty, unmounting and removing...")
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print(f"Error during unmount: {e}")
    # os.system("fusermount -uz /content/drive") # Alternative for force unmount if drive.flush_and_unmount fails
    shutil.rmtree("/content/drive", ignore_errors=True)
    os.makedirs("/content/drive", exist_ok=True)

drive.mount("/content/drive", force_remount=True)

PROJECT_NAME = "AIGC_Fake_Detection"
GITHUB_REPO = "https://github.com/IanJ332/AIGC_Fake_Detection.git"
BRANCH = "main"

DRIVE_ROOT = Path("/content/drive/MyDrive/AIGC")
DATA_DIR = DRIVE_ROOT / "Data_V2"
NOTEBOOK_DIR = DRIVE_ROOT / "Notebook"
REPO_DIR = Path("/content/AIGC_Fake_Detection")
MANIFEST_PATH = "corpus/manifest.csv"

RUN_DOWNLOAD = True
RUN_PARSE = True
FORCE_REBUILD = False

USE_BACKFILL_BUILDER = True
TARGET_PARSED = 100
MAX_CANDIDATES = 300

print("BRANCH =", BRANCH)
print("DATA_DIR =", DATA_DIR)
print("MANIFEST_PATH =", MANIFEST_PATH)

Mount point /content/drive not empty, unmounting and removing...
Mounted at /content/drive
BRANCH = day9-datav2-notebook-hardfix
DATA_DIR = /content/drive/MyDrive/AIGC/Data_V2
MANIFEST_PATH = corpus/manifest.csv


In [3]:
subdirs = [
    "pdfs", "parsed", "sections", "tables", "registry",
    "download_logs", "parse_logs", "probes", "reports",
    "checkpoints", "tei_xml"
]

if FORCE_REBUILD and DATA_DIR.exists():
    print(f"FORCE_REBUILD=True. Removing {DATA_DIR}")
    shutil.rmtree(DATA_DIR)

for name in subdirs:
    (DATA_DIR / name).mkdir(parents=True, exist_ok=True)

print("Data_V2 folders ready:")
for name in subdirs:
    print(" -", DATA_DIR / name)

Data_V2 folders ready:
 - /content/drive/MyDrive/AIGC/Data_V2/pdfs
 - /content/drive/MyDrive/AIGC/Data_V2/parsed
 - /content/drive/MyDrive/AIGC/Data_V2/sections
 - /content/drive/MyDrive/AIGC/Data_V2/tables
 - /content/drive/MyDrive/AIGC/Data_V2/registry
 - /content/drive/MyDrive/AIGC/Data_V2/download_logs
 - /content/drive/MyDrive/AIGC/Data_V2/parse_logs
 - /content/drive/MyDrive/AIGC/Data_V2/probes
 - /content/drive/MyDrive/AIGC/Data_V2/reports
 - /content/drive/MyDrive/AIGC/Data_V2/checkpoints
 - /content/drive/MyDrive/AIGC/Data_V2/tei_xml


In [4]:
!rm -rf /content/AIGC_Fake_Detection
!git clone -b main https://github.com/IanJ332/AIGC_Fake_Detection.git /content/AIGC_Fake_Detection
!cd /content/AIGC_Fake_Detection && git branch --show-current
!cd /content/AIGC_Fake_Detection && git log --oneline -3
!cd /content/AIGC_Fake_Detection && pip install -r requirements.txt

Cloning into '/content/AIGC_Fake_Detection'...
remote: Enumerating objects: 420, done.
remote: Counting objects: 100% (420/420), done.
remote: Compressing objects: 100% (308/308), done.
remote: Total 420 (delta 230), reused 292 (delta 107), pack-reused 0 (from 0)
Receiving objects: 100% (420/420), 1.30 MiB | 5.49 MiB/s, done.
Resolving deltas: 100% (230/230), done.
day9-datav2-notebook-hardfix
c21c0e7 (HEAD -> day9-datav2-notebook-hardfix, origin/day9-datav2-notebook-hardfix) Day 9: Fix candidate paper_id mapping in backfill pipeline
203aabc Day 9: Broaden corpus scope and add OA candidate expansion
834f77d Day 9: Fix Data_V2 directory creation and track manifest candidates


In [5]:
manifest_src = REPO_DIR / MANIFEST_PATH
assert manifest_src.exists(), f"Manifest not found: {manifest_src}"

import pandas as pd
manifest_df = pd.read_csv(manifest_src)
manifest_rows = len(manifest_df)
print("Manifest rows:", manifest_rows)

shutil.copy(manifest_src, DATA_DIR / "registry" / "manifest.csv")
shutil.copy(manifest_src, DATA_DIR / "registry" / "manifest_100.csv")
print("Manifest copied to Data_V2 registry.")

Manifest rows: 100
Manifest copied to Data_V2 registry.


In [6]:
candidate_pool_path = REPO_DIR / "corpus" / "manifest_candidates_v2.csv"
if not candidate_pool_path.exists():
    candidate_pool_path = REPO_DIR / "artifacts" / "manifests" / "manifest_candidates.csv"

if USE_BACKFILL_BUILDER and candidate_pool_path.exists():
    print("Running Backfill Builder...")
    !cd /content/AIGC_Fake_Detection && python scripts/build_executable_corpus.py \
      --seed-manifest corpus/manifest.csv \
      --candidate-pool {candidate_pool_path.relative_to(REPO_DIR)} \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2 \
      --target-parsed {TARGET_PARSED} \
      --max-candidates {MAX_CANDIDATES} \
      --batch-size 25 \
      --delay 1.5
else:
    print("Backfill disabled or candidate pool missing. Running static download...")
    if not candidate_pool_path.exists():
        with open(REPO_DIR / "docs" / "day9_backfill_unavailable.md", "w") as f:
            f.write("# Day 9 Backfill Unavailable\nCandidate pool missing.\n")
    !cd /content/AIGC_Fake_Detection && python scripts/download_corpus.py \
      --manifest corpus/manifest.csv \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2

    !cd /content/AIGC_Fake_Detection && python -m src.parse.parse_pdfs \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2

    !cd /content/AIGC_Fake_Detection && python -m src.parse.segment_sections \
      --data-dir /content/drive/MyDrive/AIGC/Data_V2

    !cd /content/AIGC_Fake_Detection && python -m src.parse.extract_table_candidates \
      --sections /content/drive/MyDrive/AIGC/Data_V2/sections/sections.jsonl \
      --out /content/drive/MyDrive/AIGC/Data_V2/tables/table_candidates.jsonl

Running Backfill Builder...

--- Batch 1 ---
Candidate paper_id null count: 0
Candidate paper_id duplicate count: 0
[  1/25] ✓ P011: success — already_exists
[  2/25] ✗ P001: blocked — not_a_pdf_response
[  3/25] ✗ P073: blocked — http_403
[  4/25] – P013: skipped — likely_paywalled
[  5/25] – P074: skipped — likely_paywalled
[  6/25] – P012: skipped — likely_paywalled
[  7/25] ✗ P075: blocked — http_403
[  8/25] – P018: skipped — likely_paywalled
[  9/25] – P014: skipped — likely_paywalled
[ 10/25] – P015: skipped — likely_paywalled
[ 11/25] ✓ P019: success — already_exists
[ 12/25] ✓ P016: success — already_exists
[ 13/25] ✓ P017: success — already_exists
[ 14/25] ✗ P077: blocked — http_403
[ 15/25] – P002: skipped — likely_paywalled
[ 16/25] ✓ P079: success — already_exists
[ 17/25] ✓ P080: success — already_exists
[ 18/25] ✓ P081: success — already_exists
[ 19/25] ✗ P082: blocked — http_403
[ 20/25] ✓ P020: success — already_exists
[ 21/25] ✗ P083: blocked — http_403
[ 22/25] – P02

In [7]:
import pandas as pd, glob, os

pdf_files = sorted((DATA_DIR / "pdfs").glob("*.pdf"))
zero_byte_pdfs = [p for p in pdf_files if p.stat().st_size == 0]
registry_path = DATA_DIR / "registry" / "document_registry.csv"

print("PDF count:", len(pdf_files))
print("Zero-byte PDFs:", len(zero_byte_pdfs))
print("Registry exists:", registry_path.exists())

if registry_path.exists():
    reg = pd.read_csv(registry_path)
    print("Registry rows:", len(reg))
    if "download_status" in reg.columns:
        print(reg["download_status"].value_counts(dropna=False).to_string())

PDF count: 117
Zero-byte PDFs: 0
Registry exists: True
Registry rows: 200
download_status
success    117
blocked     44
skipped     38
failed       1


In [8]:
import json, glob, os
from pathlib import Path

pdf_count = len(list((DATA_DIR / "pdfs").glob("*.pdf")))
parsed_json_count = len(list((DATA_DIR / "parsed").glob("*.json")))
zero_byte_count = len([p for p in (DATA_DIR / "pdfs").glob("*.pdf") if p.stat().st_size == 0])
sections_exists = (DATA_DIR / "sections" / "sections.jsonl").exists()
tables_exists = (DATA_DIR / "tables" / "table_candidates.jsonl").exists()
registry_exists = (DATA_DIR / "registry" / "document_registry.csv").exists()
parse_registry_exists = (DATA_DIR / "registry" / "parse_registry.csv").exists()

status = "BLOCKED"
if (DATA_DIR / "probes" / "executable_corpus_status.json").exists():
    with open(DATA_DIR / "probes" / "executable_corpus_status.json", "r") as f:
        st = json.load(f)
        status = st.get("status", "BLOCKED")
else:
    if parsed_json_count >= 95:
        status = "READY_FOR_NOTEBOOK_02"
    elif 70 <= parsed_json_count < 95:
        status = "CAUTION_READY_FOR_NOTEBOOK_02"

summary = {
    "branch": BRANCH,
    "data_dir": str(DATA_DIR),
    "manifest_path": MANIFEST_PATH,
    "manifest_rows": manifest_rows,
    "pdf_count": pdf_count,
    "zero_byte_pdf_count": zero_byte_count,
    "parsed_json_count": parsed_json_count,
    "sections_exists": sections_exists,
    "tables_exists": tables_exists,
    "registry_exists": registry_exists,
    "parse_registry_exists": parse_registry_exists,
    "final_status": status,
}

print(json.dumps(summary, indent=2))

with open(DATA_DIR / "probes" / "day9_data_status.json", "w") as f:
    json.dump(summary, f, indent=2)

with open(DATA_DIR / "reports" / "day9_data_sync_report.md", "w") as f:
    f.write("# Day 9 Data Sync Report\n\n")
    for k, v in summary.items():
        f.write(f"- **{k}**: {v}\n")

print("Saved:")
print(DATA_DIR / "probes" / "day9_data_status.json")
print(DATA_DIR / "reports" / "day9_data_sync_report.md")
print("FINAL STATUS:", status)

{
  "branch": "day9-datav2-notebook-hardfix",
  "data_dir": "/content/drive/MyDrive/AIGC/Data_V2",
  "manifest_path": "corpus/manifest.csv",
  "manifest_rows": 100,
  "pdf_count": 117,
  "zero_byte_pdf_count": 0,
  "parsed_json_count": 117,
  "sections_exists": true,
  "tables_exists": true,
  "registry_exists": true,
  "parse_registry_exists": true,
  "final_status": "BLOCKED"
}
Saved:
/content/drive/MyDrive/AIGC/Data_V2/probes/day9_data_status.json
/content/drive/MyDrive/AIGC/Data_V2/reports/day9_data_sync_report.md
FINAL STATUS: BLOCKED


In [9]:
from pathlib import Path
import pandas as pd, json

DATA_DIR = Path("/content/drive/MyDrive/AIGC/Data_V2")

print("PDF files:", len(list((DATA_DIR / "pdfs").glob("*.pdf"))))
print("Parsed JSON:", len(list((DATA_DIR / "parsed").glob("*.json"))))

reg_path = DATA_DIR / "registry" / "document_registry.csv"
print("Registry exists:", reg_path.exists())

if reg_path.exists():
    reg = pd.read_csv(reg_path)
    print("Registry rows:", len(reg))
    print("\nDownload status:")
    print(reg["download_status"].value_counts(dropna=False).to_string())
    print("\nDownload reason:")
    print(reg["download_reason"].value_counts(dropna=False).head(30).to_string())
    print("\nDownloaded true:")
    print(reg["pdf_downloaded"].value_counts(dropna=False).to_string())
    print("\nRows with pdf_downloaded=True:")
    print(reg[reg["pdf_downloaded"] == True][["paper_id", "title", "download_status", "download_reason", "pdf_path"]].head(20).to_string())

backfill_status = DATA_DIR / "probes" / "executable_corpus_status.json"
print("\nBackfill status exists:", backfill_status.exists())
if backfill_status.exists():
    print(backfill_status.read_text())

backfill_report = DATA_DIR / "download_logs" / "backfill_report.md"
print("\nBackfill report exists:", backfill_report.exists())
if backfill_report.exists():
    print(backfill_report.read_text()[:2000])

PDF files: 117
Parsed JSON: 117
Registry exists: True
Registry rows: 200

Download status:
download_status
success    117
blocked     44
skipped     38
failed       1

Download reason:
download_reason
already_exists                                                                                                              99
http_403                                                                                                                    42
likely_paywalled                                                                                                            38
NaN                                                                                                                         18
not_a_pdf_response                                                                                                           2
HTTPSConnectionPool(host='ijas.uodiyala.edu.iq', port=443): Max retries exceeded with url: /index.php/IJAS/article/downl     1

Downloaded true:
pdf_downloaded
True